In [23]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import sys
import yaml
import json
from datasets import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
import gc
from datasets import Dataset
from datetime import datetime
from sklearn.metrics import roc_curve
from scipy.interpolate import interp1d
import zlib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from rouge_score import rouge_scorer
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

DEBUG = True

# Setup directories
curr_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
print(f"Current directory: {curr_dir}")
PROJECT_DIR = os.path.abspath(os.path.join(curr_dir, '..', '..'))
print(f"Project directory: {PROJECT_DIR}")
Unlearn_Simple_DIR = os.path.join(PROJECT_DIR, 'Unlearn-Simple')
print(f"Unlearn_Simple_DIR: {Unlearn_Simple_DIR}")

TOFU_DIR = os.path.join(Unlearn_Simple_DIR, 'TOFU')
MUSE_DIR = os.path.join(Unlearn_Simple_DIR, 'MUSE')
WMDP_DIR = os.path.join(Unlearn_Simple_DIR, 'WMDP')

# Add necessary paths
sys.path.append(PROJECT_DIR)
sys.path.append(Unlearn_Simple_DIR)
sys.path.append(os.path.join(TOFU_DIR))
sys.path.append(os.path.join(MUSE_DIR))
sys.path.append(os.path.join(WMDP_DIR))
sys.path.append(os.path.join(MUSE_DIR, 'src'))
sys.path.append(os.path.join(PROJECT_DIR, 'src'))

# Helper function for GPU memory cleanup
def cleanup_gpu_memory():
    """Clean up GPU memory and garbage collect."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        
# Clean up memory after each iteration
cleanup_gpu_memory()

print("Setup complete!")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Current directory: /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/Unlearn-Simple/notbooks
Project directory: /home/liranc6/W25/adversarial-attacks-on-deep-learning/project
Unlearn_Simple_DIR: /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/Unlearn-Simple
Setup complete!
CUDA available: True
GPU: NVIDIA A40
GPU memory: 47.3 GB


In [24]:
# Import evaluation modules
import MUSE.src.eval_with_ILL as eval_with_ILL
from src.input_loss_landscape.utils import new_ILL_eval
import src.utils as project_utils

# Define all model configurations for experiments
MODEL_CONFIGS = {
    # WMDP models (LLaMA-3-8B with different unlearning methods)
    "wmdp_models": [
        "LLM-GAT/llama-3-8b-instruct-elm-checkpoint-8",
        # "LLM-GAT/llama-3-8b-instruct-graddiff-checkpoint-8", 
        # "LLM-GAT/llama-3-8b-instruct-tar-checkpoint-8",
        # "LLM-GAT/llama-3-8b-instruct-pbj-checkpoint-8",
        # "LLM-GAT/llama-3-8b-instruct-rmu-lat-checkpoint-8",
        # "LLM-GAT/llama-3-8b-instruct-rmu-checkpoint-8",
        # "LLM-GAT/llama-3-8b-instruct-rr-checkpoint-8",
        # "LLM-GAT/llama-3-8b-instruct-repnoise-checkpoint-8"
    ],
    
    # TOFU models (LLaMA-2-7B)
    "tofu_models": [
        "OPTML-Group/SimNPO-TOFU-forget05-Llama-2-7b-chat",
        "OPTML-Group/SimNPO-TOFU-forget10-Llama-2-7b-chat"
    ],
    
    # MUSE models (LLaMA-2-7B)
    "muse_models": [
        "OPTML-Group/SimNPO-MUSE-News-llama-2-7b",
        "OPTML-Group/SimNPO-MUSE-Books-llama-2-7b"
    ],
    
    # Additional WMDP model
    # "wmdp_zephyr": [
    #     "OPTML-Group/SimNPO-WMDP-zephyr-7b-beta"
    # ]
}

# Define unlearning method mapping
UNLEARNING_METHODS = {
    "elm": "ELM",
    "graddiff": "GradDiff", 
    "tar": "TAR",
    "pbj": "PBJ",
    "rmu-lat": "RMU-LAT",
    "rmu": "RMU",
    "rr": "RR", 
    "repnoise": "RepNoise",
    "simnpo": "SimNPO"
}

# Model families for results aggregation
MODEL_FAMILIES = {
    "LLaMA-3-8B": MODEL_CONFIGS["wmdp_models"],
    "LLaMA-2-7B": MODEL_CONFIGS["tofu_models"] + MODEL_CONFIGS["muse_models"],
    # "Zephyr-7B": MODEL_CONFIGS["wmdp_zephyr"]
}

print("Model configurations defined:")
for family, models in MODEL_FAMILIES.items():
    print(f"  {family}: {len(models)} models")
print(f"Total models to evaluate: {sum(len(models) for models in MODEL_FAMILIES.values())}")

Model configurations defined:
  LLaMA-3-8B: 1 models
  LLaMA-2-7B: 4 models
Total models to evaluate: 5


In [25]:
def load_tofu_data(subset_size=50):
    """Load TOFU forget, retain, and holdout datasets"""
    print("Loading TOFU data...")

    def tofu_hf_to_dict(split):
        # Each item has 'question' and 'answer'
        return load_dataset("locuslab/TOFU", split, split='train')

    forget_prc = 10
    forget_data = tofu_hf_to_dict(f'forget{forget_prc}')
    retain_data = tofu_hf_to_dict(f'retain{100-forget_prc}')
    holdout_data = tofu_hf_to_dict('holdout10')
    
    # Limit dataset sizes
    forget_data = forget_data.select(range(min(subset_size, len(forget_data))))
    retain_data = retain_data.select(range(min(subset_size, len(retain_data))))
    holdout_data = holdout_data.select(range(min(subset_size, len(holdout_data))))
    
    print(f"TOFU data loaded: {len(forget_data)} forget, {len(retain_data)} retain, {len(holdout_data)} holdout")
    return forget_data, retain_data, holdout_data

def load_wmdp_data(subset_size=50):
    """Load WMDP forget, retain, and holdout datasets"""
    print("Loading WMDP data...")

    # Load WMDP datasets
    forget_data = load_dataset("cais/wmdp-bio-forget-corpus", cache_dir="./.cache", split='train')
    retain_data = load_dataset("cais/wmdp-corpora", "bio-retain-corpus", cache_dir="./.cache")['train']
    raw_holdout_data = load_dataset("cais/wmdp", "wmdp-bio", cache_dir="./.cache", split='test')
    
    # Convert holdout format
    def convert_holdout(example):
        return {'question': example['question'], 'answer': example['choices'][example['answer']]}
    
    holdout_data = raw_holdout_data.map(convert_holdout, remove_columns=['choices'])
    
    # Limit dataset sizes
    forget_data = forget_data.select(range(min(subset_size, len(forget_data))))
    retain_data = retain_data.select(range(min(subset_size, len(retain_data))))
    holdout_data = holdout_data.select(range(min(subset_size, len(holdout_data))))
    
    print(f"WMDP data loaded: {len(forget_data)} forget, {len(retain_data)} retain, {len(holdout_data)} holdout")
    return forget_data, retain_data, holdout_data

def load_muse_data(subset_size=50):
    """Load MUSE forget, retain, and holdout datasets"""  
    print("Loading MUSE data...")
    
    # Load MUSE datasets
    raw_or_privleak = 'raw'

    forget_file = os.path.join(MUSE_DIR, f'data/news/{raw_or_privleak}/forget.json')
    if raw_or_privleak == 'raw':
        retain_file = os.path.join(MUSE_DIR, f'data/news/{raw_or_privleak}/retain1.json')
    else:
        retain_file = os.path.join(MUSE_DIR, f'data/news/{raw_or_privleak}/retain.json')
    holdout_file = os.path.join(MUSE_DIR, f'data/news/{raw_or_privleak}/holdout.json')

    forget_data = eval_with_ILL.read_json(forget_file)
    retain_data = eval_with_ILL.read_json(retain_file)
    holdout_data = eval_with_ILL.read_json(holdout_file)
    
    # Convert to HuggingFace Dataset
    forget_data = Dataset.from_dict({'text': forget_data})
    retain_data = Dataset.from_dict({'text': retain_data})
    holdout_data = Dataset.from_dict({'text': holdout_data})

    # Limit dataset sizes
    forget_data = forget_data.select(range(min(subset_size, len(forget_data))))
    retain_data = retain_data.select(range(min(subset_size, len(retain_data))))
    holdout_data = holdout_data.select(range(min(subset_size, len(holdout_data))))
    
    print(f"MUSE data loaded: {len(forget_data)} forget, {len(retain_data)} retain, {len(holdout_data)} holdout")
    return forget_data, retain_data, holdout_data

# Define data loading functions for each benchmark
DATA_LOADERS = {
    "TOFU": load_tofu_data,
    "WMDP": load_wmdp_data,
    "MUSE": load_muse_data
}

print("Data loading functions defined for all benchmarks")

Data loading functions defined for all benchmarks


In [26]:
# Utility functions for dataset processing
def identify_prompt_column_type(example):
    if 'text' in example:
        return 'text'
    elif 'question' in example and 'answer' in example:
        return ['question', 'answer']
    else:
        raise ValueError("Dataset must contain either 'text' or both 'question' and 'answer' fields.")

def extract_texts_from_dataset(dataset):
    """
    Extract texts from a dataset using automatic column type detection.
    
    Args:
        dataset: List or Dataset of examples (e.g., forget_data, retain_data).
    
    Returns:
        list: List of extracted text strings.
    """
    prompt_column = identify_prompt_column_type(dataset[0])
    if isinstance(prompt_column, str):
        texts = dataset[prompt_column]
    else:
        texts = dataset[prompt_column[0]]  # Just use question for simplicity
        
    return texts

def filter_examples_by_num_tokens(example, prompt_column, tokenizer, num_min_tokens=5, max_new_tokens=50):
    example['valid_example'] = False  # default to invalid

    if isinstance(prompt_column, str):
        text = example.get(prompt_column, '')
        tokens = tokenizer.tokenize(text)

        if len(tokens) < num_min_tokens:
            return example

        if len(tokens) > max_new_tokens:
            random_num_tokens = np.random.randint(num_min_tokens, max_new_tokens)
            text = tokenizer.convert_tokens_to_string(tokens[:random_num_tokens])
            example[prompt_column] = text

        example['valid_example'] = True
        return example

    elif isinstance(prompt_column, list) and len(prompt_column) == 2:
        question = example.get(prompt_column[0], '')
        answer = example.get(prompt_column[1], '')
        text = question + " " + answer

        # text_tokens = tokenizer.tokenize(text)
        question_tokens = tokenizer.tokenize(question)
        answer_tokens = tokenizer.tokenize(answer)

        if len(question_tokens) < num_min_tokens:
            return example

        if len(text) > max_new_tokens:
            ans_max_tokens = max_new_tokens - len(question_tokens)
            if ans_max_tokens < num_min_tokens:
                return example
            random_num_tokens = np.random.randint(num_min_tokens, ans_max_tokens)
            answer = tokenizer.convert_tokens_to_string(answer_tokens[:random_num_tokens])
            example[prompt_column[1]] = answer  # update only the answer

        # Question is not modified
        example['valid_example'] = True
        return example

    # Unsupported format
    return example

In [27]:
def get_text(ex):
    """Automatically extract text from an example based on its format."""
    try:
        prompt_type = identify_prompt_column_type(ex)
        if prompt_type == 'text':
            return ex['text']
        elif isinstance(prompt_type, list):
            return f"{ex.get('question', '')} {ex.get('answer', '')}".strip()
    except ValueError:
        # Fallback: try 'text', then 'question' + 'answer', then empty string
        return ex.get('text', f"{ex.get('question', '')} {ex.get('answer', '')}".strip())

def compute_zlib_compression_auc(forget_data, retain_data, holdout_data, model, tokenizer):
    """
    Compute Zlib compression-based AUC for dataset distinctions using model-generated responses.
    Lower compression ratio indicates higher entropy (potentially "forgotten" data).
    """
    def get_compression_ratio(text):
        compressed = zlib.compress(text.encode('utf-8'))
        return len(compressed) / len(text.encode('utf-8'))
    
    def generate_response(prompt, model, tokenizer, max_new_tokens=50):
        """Generate a response for a given prompt."""
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        response = None
        max_attempts = 10
        attempt = 0
        while not response and attempt < max_attempts:
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
                response = tokenizer.decode(outputs[0], skip_special_tokens=True).replace(prompt, '').strip()
            attempt += 1
        if not response:
            response = ""  # Fallback to empty response if generation fails
        return response
    
    # Generate responses for each dataset
    forget_responses = []
    retain_responses = []
    holdout_responses = []
    
    for ex in tqdm(forget_data, desc="Processing forget data", total=len(forget_data)):
        prompt = get_text(ex)  # Use existing helper to get input text
        response = generate_response(prompt, model, tokenizer)
        forget_responses.append(response)
    
    for ex in tqdm(retain_data, desc="Processing retain data", total=len(retain_data)):
        prompt = get_text(ex)
        response = generate_response(prompt, model, tokenizer)
        retain_responses.append(response)
    
    for ex in tqdm(holdout_data, desc="Processing holdout data", total=len(holdout_data)):
        prompt = get_text(ex)
        response = generate_response(prompt, model, tokenizer)
        holdout_responses.append(response)
        
    # remove empty responses
    forget_responses = [resp for resp in forget_responses if resp]
    retain_responses = [resp for resp in retain_responses if resp]
    holdout_responses = [resp for resp in holdout_responses if resp]

    # subset to the same size
    min_size = min(len(forget_responses), len(retain_responses), len(holdout_responses))
    forget_responses = forget_responses[:min_size]
    retain_responses = retain_responses[:min_size]
    holdout_responses = holdout_responses[:min_size]

    # Compute ratios on generated responses
    forget_ratios = [get_compression_ratio(resp) for resp in forget_responses]
    retain_ratios = [get_compression_ratio(resp) for resp in retain_responses]
    holdout_ratios = [get_compression_ratio(resp) for resp in holdout_responses]

    # Retain vs All (retain=1, others=0)
    X_ra = np.array(retain_ratios + forget_ratios + holdout_ratios).reshape(-1, 1)
    y_ra = np.array([1] * len(retain_ratios) + [0] * (len(forget_ratios) + len(holdout_ratios)))
    clf_ra = LogisticRegression().fit(X_ra, y_ra)
    retain_vs_all_auc = roc_auc_score(y_ra, clf_ra.predict_proba(X_ra)[:, 1])

    # Forget vs All (forget=1, others=0)
    X_fa = np.array(forget_ratios + retain_ratios + holdout_ratios).reshape(-1, 1)
    y_fa = np.array([1] * len(forget_ratios) + [0] * (len(retain_ratios) + len(holdout_ratios)))
    clf_fa = LogisticRegression().fit(X_fa, y_fa)
    forget_vs_all_auc = roc_auc_score(y_fa, clf_fa.predict_proba(X_fa)[:, 1])

    # Holdout vs All (holdout=1, others=0)
    X_ha = np.array(holdout_ratios + retain_ratios + forget_ratios).reshape(-1, 1)
    y_ha = np.array([1] * len(holdout_ratios) + [0] * (len(retain_ratios) + len(forget_ratios)))
    clf_ha = LogisticRegression().fit(X_ha, y_ha)
    holdout_vs_all_auc = roc_auc_score(y_ha, clf_ha.predict_proba(X_ha)[:, 1])

    # Multiclass AUC: 3-class classifier (forget=0, retain=1, holdout=2)
    X_multi = np.array(forget_ratios + retain_ratios + holdout_ratios).reshape(-1, 1)
    y_multi = np.array([0] * len(forget_ratios) + [1] * len(retain_ratios) + [2] * len(holdout_ratios))
    clf_multi = LogisticRegression(multi_class='ovr').fit(X_multi, y_multi)
    multi_class_auc = roc_auc_score(y_multi, clf_multi.predict_proba(X_multi), multi_class='ovr')

    # Updated return with new keys
    return {
        'retain_vs_all_auc': retain_vs_all_auc,
        'forget_vs_all_auc': forget_vs_all_auc,
        'holdout_vs_all_auc': holdout_vs_all_auc,
        'multi_class_auc': multi_class_auc
    }

def compute_min_k_percent_auc(forget_data, retain_data, holdout_data, model, tokenizer, k_percent=0.1):
    """
    Compute MIN-K%++ based AUC using top-k% token probabilities.
    """
    def get_top_k_prob(text, k_percent):
        # Retry until a non-empty response is obtained
        max_attempts = 10
        attempt = 0
        top_probs = None
        while attempt < max_attempts:
            inputs = tokenizer(text, return_tensors='pt').to(model.device)
            with torch.no_grad():
                outputs = model(**inputs)
                probs = torch.softmax(outputs.logits, dim=-1)
                vocab_size = probs.shape[-1]
                if vocab_size == 0:
                    attempt += 1
                    continue
                top_k = max(1, int(k_percent * vocab_size))
                top_probs = torch.topk(probs, top_k, dim=-1).values.mean(dim=-1).cpu().numpy()
                if top_probs.size == 0 or np.all(top_probs == 0):
                    attempt += 1
                    continue
                break
            attempt += 1
        if top_probs is None or top_probs.size == 0:
            return np.nan
        return top_probs.mean()  # Average over sequence
    
    # Extract texts using automatic detection (updated for consistency)
    forget_texts = extract_texts_from_dataset(forget_data)
    retain_texts = extract_texts_from_dataset(retain_data)
    holdout_texts = extract_texts_from_dataset(holdout_data)
    
    # Compute scores
    forget_scores = [get_top_k_prob(t, k_percent) for t in forget_texts]
    retain_scores = [get_top_k_prob(t, k_percent) for t in retain_texts]
    holdout_scores = [get_top_k_prob(t, k_percent) for t in holdout_texts]
    
    # remove np.nan scores
    forget_scores = [s for s in forget_scores if not np.isnan(s)]
    retain_scores = [s for s in retain_scores if not np.isnan(s)]
    holdout_scores = [s for s in holdout_scores if not np.isnan(s)]
    # subset to the same size
    min_size = min(len(forget_scores), len(retain_scores), len(holdout_scores))
    forget_scores = forget_scores[:min_size]
    retain_scores = retain_scores[:min_size]
    holdout_scores = holdout_scores[:min_size]

    # Retain vs All (retain=1, others=0)
    X_ra = np.array(retain_scores + forget_scores + holdout_scores).reshape(-1, 1)
    y_ra = np.array([1] * len(retain_scores) + [0] * (len(forget_scores) + len(holdout_scores)))
    clf_ra = LogisticRegression().fit(X_ra, y_ra)
    retain_vs_all_auc = roc_auc_score(y_ra, clf_ra.predict_proba(X_ra)[:, 1])

    # Forget vs All (forget=1, others=0)
    X_fa = np.array(forget_scores + retain_scores + holdout_scores).reshape(-1, 1)
    y_fa = np.array([1] * len(forget_scores) + [0] * (len(retain_scores) + len(holdout_scores)))
    clf_fa = LogisticRegression().fit(X_fa, y_fa)
    forget_vs_all_auc = roc_auc_score(y_fa, clf_fa.predict_proba(X_fa)[:, 1])

    # Holdout vs All (holdout=1, others=0)
    X_ha = np.array(holdout_scores + retain_scores + forget_scores).reshape(-1, 1)
    y_ha = np.array([1] * len(holdout_scores) + [0] * (len(retain_scores) + len(forget_scores)))
    clf_ha = LogisticRegression().fit(X_ha, y_ha)
    holdout_vs_all_auc = roc_auc_score(y_ha, clf_ha.predict_proba(X_ha)[:, 1])

    # Multiclass AUC: 3-class classifier (forget=0, retain=1, holdout=2)
    X_multi = np.array(forget_scores + retain_scores + holdout_scores).reshape(-1, 1)
    y_multi = np.array([0] * len(forget_scores) + [1] * len(retain_scores) + [2] * len(holdout_scores))
    clf_multi = LogisticRegression(multi_class='ovr').fit(X_multi, y_multi)
    multi_class_auc = roc_auc_score(y_multi, clf_multi.predict_proba(X_multi), multi_class='ovr')

    # Updated return with new keys
    return {
        'retain_vs_all_auc': retain_vs_all_auc,
        'forget_vs_all_auc': forget_vs_all_auc,
        'holdout_vs_all_auc': holdout_vs_all_auc,
        'multi_class_auc': multi_class_auc
    }

def compute_rouge_l_f1_auc(forget_data, retain_data, holdout_data, model, tokenizer):
    """
    Compute ROUGE-L F1 based AUC using generated vs. reference answers.
    """
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    
    def get_rouge_score(question, ref_answer, model, tokenizer):
        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        
        inputs = tokenizer(question, return_tensors='pt').to(model.device)
        input_ids = inputs['input_ids'][0]
        
        
        if not ref_answer.strip():
            # take 0.3 of the question as fallback
            len_ans_ids = int(0.3 * len(input_ids))
            ref_answer_ids = input_ids[-len_ans_ids:]
            ref_answer = tokenizer.decode(ref_answer_ids, skip_special_tokens=True).strip()
            # Remove fallback answer from prompt for generation
            inputs = {
                'input_ids': input_ids[:-len_ans_ids].unsqueeze(0),
                'attention_mask': inputs['attention_mask'][0][:-len_ans_ids].unsqueeze(0)
            }
        
        gen_answer = ""
        max_attempts = 10
        attempt = 0
        while not gen_answer and attempt < max_attempts:
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=300, do_sample=False, pad_token_id=tokenizer.eos_token_id)
                gen_answer = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True).strip()
            attempt += 1
        if not gen_answer:
            return None  # Fallback to None if generation fails
        score = scorer.score(ref_answer, gen_answer)['rougeL'].fmeasure
        return score
    
    # Helper to extract question-answer pairs using modular functions
    def extract_qa_pairs(dataset):
        if not dataset:
            return []
        try:
            prompt_column = identify_prompt_column_type(dataset[0])
            qa_pairs = []
            for ex in dataset:
                if isinstance(prompt_column, list) and len(prompt_column) == 2:  # 'question' and 'answer'
                    question = ex.get(prompt_column[0], "")
                    answer = ex.get(prompt_column[1], "")
                elif prompt_column == 'text':  # 'text' case: treat as question, empty answer
                    question = ex.get('text', "")
                    answer = ""  # No reference answer for text-based datasets
                else:
                    question = ""
                    answer = ""
                qa_pairs.append((question, answer))
            return qa_pairs
        except ValueError as e:
            print(f"Warning: {e}. Returning empty QA pairs.")
            return []
    
    # Extract QA pairs for each dataset
    forget_qa = extract_qa_pairs(forget_data)
    retain_qa = extract_qa_pairs(retain_data)
    holdout_qa = extract_qa_pairs(holdout_data)
    
    # Compute scores
    forget_scores = [get_rouge_score(q, a, model, tokenizer) for q, a in tqdm(forget_qa, desc="ROUGE forget", total=len(forget_qa))]
    retain_scores = [get_rouge_score(q, a, model, tokenizer) for q, a in tqdm(retain_qa, desc="ROUGE retain", total=len(retain_qa))]
    holdout_scores = [get_rouge_score(q, a, model, tokenizer) for q, a in tqdm(holdout_qa, desc="ROUGE holdout", total=len(holdout_qa))]
    
    # remove None scores
    forget_scores = [s for s in forget_scores if s is not None]
    retain_scores = [s for s in retain_scores if s is not None]
    holdout_scores = [s for s in holdout_scores if s is not None]
    # subset to the same size
    min_size = min(len(forget_scores), len(retain_scores), len(holdout_scores))
    forget_scores = forget_scores[:min_size]
    retain_scores = retain_scores[:min_size]
    holdout_scores = holdout_scores[:min_size]
    
    # Retain vs All (retain=1, others=0)
    X_ra = np.array(retain_scores + forget_scores + holdout_scores).reshape(-1, 1)
    y_ra = np.array([1] * len(retain_scores) + [0] * (len(forget_scores) + len(holdout_scores)))
    clf_ra = LogisticRegression().fit(X_ra, y_ra)
    retain_vs_all_auc = roc_auc_score(y_ra, clf_ra.predict_proba(X_ra)[:, 1])

    # Forget vs All (forget=1, others=0)
    X_fa = np.array(forget_scores + retain_scores + holdout_scores).reshape(-1, 1)
    y_fa = np.array([1] * len(forget_scores) + [0] * (len(retain_scores) + len(holdout_scores)))
    clf_fa = LogisticRegression().fit(X_fa, y_fa)
    forget_vs_all_auc = roc_auc_score(y_fa, clf_fa.predict_proba(X_fa)[:, 1])

    # Holdout vs All (holdout=1, others=0)
    X_ha = np.array(holdout_scores + retain_scores + forget_scores).reshape(-1, 1)
    y_ha = np.array([1] * len(holdout_scores) + [0] * (len(retain_scores) + len(forget_scores)))
    clf_ha = LogisticRegression().fit(X_ha, y_ha)
    holdout_vs_all_auc = roc_auc_score(y_ha, clf_ha.predict_proba(X_ha)[:, 1])

    # Multiclass AUC: 3-class classifier (forget=0, retain=1, holdout=2)
    X_multi = np.array(forget_scores + retain_scores + holdout_scores).reshape(-1, 1)
    y_multi = np.array([0] * len(forget_scores) + [1] * len(retain_scores) + [2] * len(holdout_scores))
    clf_multi = LogisticRegression(multi_class='ovr').fit(X_multi, y_multi)
    multi_class_auc = roc_auc_score(y_multi, clf_multi.predict_proba(X_multi), multi_class='ovr')

    # Updated return with new keys
    return {
        'retain_vs_all_auc': retain_vs_all_auc,
        'forget_vs_all_auc': forget_vs_all_auc,
        'holdout_vs_all_auc': holdout_vs_all_auc,
        'multi_class_auc': multi_class_auc
    }

In [28]:
def evaluate_model_on_benchmark(model_name, benchmark_name, subset_size=30):
    """
    Evaluate a single model on a benchmark using Input Loss Landscape (ILL) analysis
    
    Args:
        model_name: HuggingFace model name/path
        benchmark_name: One of 'TOFU', 'WMDP', 'MUSE' 
        subset_size: Number of examples per dataset split
        
    Returns:
        dict: Evaluation results including AUC scores and classification metrics
    """
    print(f"\n{'='*60}")
    print(f"Evaluating {model_name} on {benchmark_name}")
    print(f"{'='*60}")
    
    try:
        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        tokenizer_name = None
        if 'llama-2' in model_name.lower():
            tokenizer_name = "meta-llama/Llama-2-7b-hf"
        elif 'llama-3' in model_name.lower():
            tokenizer_name = "meta-llama/Meta-Llama-3-8B"
        elif 'zephyr' in model_name.lower():
            tokenizer_name = "TheBloke/Zephyr-7B-Beta-GPTQ"
            
        if tokenizer_name:
            tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, cache_dir=None)
        else:
            raise ValueError(f"Unknown tokenizer for model: {model_name}")
        
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=None  # Disables caching; model is loaded directly into memory
        )
        
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        model.config.pad_token_id = tokenizer.eos_token_id
            
        print(f"Model loaded successfully on device: {next(model.parameters()).device}")
        
        # Load benchmark data
        data_loader = DATA_LOADERS[benchmark_name]
        forget_data, retain_data, holdout_data = data_loader(subset_size)
        
        # Prepare datasets for ILL evaluation
        datasets = {
            'forget': {'name': 'forget', 'data': forget_data},
            'retain': {'name': 'retain', 'data': retain_data}, 
            'holdout': {'name': 'holdout', 'data': holdout_data}
        }
        
        # Configure ILL evaluation parameters
        strategy = {'name': 'embeddings', 'peak_top_k': 20, 'n_tokens': 0.1, 'max_neighbors': 15}
        
        # Model configuration for prompt formatting
        # Model configuration for prompt formatting
        if benchmark_name == "TOFU":
            model_config_file = os.path.join(TOFU_DIR, 'config', 'model_config.yaml')
            with open(model_config_file, 'r') as f:
                model_configs_all = yaml.safe_load(f)
            # Infer model family from model_name
            if 'llama-2' in model_name.lower():
                model_family = 'llama-2-7b-chat'
            elif 'llama-3' in model_name.lower():
                model_family = 'llama-3-8b'
            elif 'zephyr' in model_name.lower():
                model_family = 'zephyr-7b-beta'
            else:
                raise ValueError(f"Unknown model family for model: {model_name}")
            model_cfg = model_configs_all.get(model_family)
            if model_cfg is None:
                # Use a minimal default config if not found
                model_cfg = {
                    'question_start_tag': '',
                    'question_end_tag': '',
                    'answer_tag': '',
                    'flash_attention2': 'false',
                    'gradient_checkpointing': 'false'
                }
                
            model_configs = {
                'question_start_tag': model_cfg['question_start_tag'],
                'question_end_tag': model_cfg['question_end_tag'],
                'answer_tag': model_cfg['answer_tag'],
                'start_of_sequence_token': "<s>"
            }
        elif benchmark_name == "WMDP":
            model_configs = {
                'question_start_tag': "### Question: ",
                'question_end_tag': "\n",
                'answer_tag': "### Answer: ",
            }
        elif benchmark_name == "MUSE":
            model_configs = {
                'question_start_tag': "",
                'question_end_tag': "",
                'answer_tag': "",
            }
        else:
            model_configs = {
                'question_start_tag': "",
                'question_end_tag': "",
                'answer_tag': "",
            }
        
        cosine_similarities_file = os.path.join(PROJECT_DIR,
                                            'models', 
                                            'distilgpt2-finetuned-wikitext2',
                                            'embeddings', 
                                            'token_knn_mapping_70_cosine.pth'
                                            )
        
        # Set up ILL evaluation kwargs
        new_ILL_eval_kwargs = {
            'model_name': model_name,
            'model': model,
            'tokenizer': tokenizer,
            'datasets': datasets,
            'prompt_column': ['question', 'answer'],
            'create_new_neighbors_file': False,  # Create neighbors on the fly
            'showplts': False,
            'cosine_similarities_file': cosine_similarities_file,
            'plots_output_dir': None,
            'strategy': strategy,
            'output_dirs': {'neighbors': f'exp_{benchmark_name.lower()}_neighbors'},
            'model_configs': model_configs,
        }
        
        print("Running ILL evaluation...")
        features_dict = new_ILL_eval(new_ILL_eval_kwargs)
        
        # Extract feature tensors
        forget_tensor = features_dict['forget']['unnormalized_features_tensor']
        retain_tensor = features_dict['retain']['unnormalized_features_tensor']
        holdout_tensor = features_dict['holdout']['unnormalized_features_tensor']
        
        # Ensure consistent sizes
        min_examples = min(len(forget_tensor), len(retain_tensor), len(holdout_tensor))
        forget_tensor = forget_tensor[:min_examples]
        retain_tensor = retain_tensor[:min_examples] 
        holdout_tensor = holdout_tensor[:min_examples]
        
        print(f"Feature tensors: forget {forget_tensor.shape}, retain {retain_tensor.shape}, holdout {holdout_tensor.shape}")
        
        # Normalize features
        norm_forget_tensor, norm_retain_tensor, norm_holdout_tensor = eval_with_ILL.normalize_features(
            forget_tensor, retain_tensor, holdout_tensor
        )
        
        # Get feature labels
        features_labels = features_dict['forget']['features_names']
        
        # Train classifiers
        print("Training predictors...")
        results, feature_importance_results = eval_with_ILL.train_predictors(
            retain_t=norm_retain_tensor,
            holdout_t=norm_holdout_tensor,
            features_labels=features_labels,
            forget_t=norm_forget_tensor
        )
        
        # Train binary comparisons
        binary_results, binary_feature_importance = eval_with_ILL.train_binary_comparisons(
            norm_retain_tensor, norm_holdout_tensor, norm_forget_tensor, features_labels
        )
        
        # Add custom binary comparisons for table generation
        def train_custom_binary_comparison(tensor1, tensor2, labels):
            """Train binary classifier between two specific tensors"""
            from sklearn.model_selection import train_test_split
            from sklearn.ensemble import RandomForestClassifier
            from sklearn.linear_model import LogisticRegression
            from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
            
            X = torch.cat([tensor1, tensor2]).numpy()
            y = torch.cat([torch.ones(len(tensor1)), torch.zeros(len(tensor2))]).numpy()
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
            
            results = {}
            for clf_type, (clf_class, params) in [
                ('logistic', (LogisticRegression, {'random_state': 42, 'max_iter': 300})), 
                ('random_forest', (RandomForestClassifier, {'random_state': 42, 'n_estimators': 100}))
            ]:
                clf = clf_class(**params).fit(X_train, y_train)
                y_pred = clf.predict(X_test)
                y_proba = clf.predict_proba(X_test)[:, 1]
                
                fpr, tpr, thresholds = roc_curve(y_test, y_proba)

                # Interpolate to find TPR at FPR = 0.01 (1%)
                if len(fpr) > 1:
                    interp_func = interp1d(fpr, tpr, kind='linear', bounds_error=False, fill_value=(tpr[0], tpr[-1]))
                    auc_at_0_1_fp = float(interp_func(0.01))
                else:
                    auc_at_0_1_fp = 0.5  # Default if insufficient data

                results[clf_type] = {
                    'accuracy': accuracy_score(y_test, y_pred),
                    'f1': f1_score(y_test, y_pred),
                    'roc_auc': roc_auc_score(y_test, y_proba),
                    'auc_at_0_1_fp': auc_at_0_1_fp  # Add this for AUC at 0.1% FP
                }
            return results
        
        # Add the specific comparisons we need for the tables
        binary_results['retain_vs_all'] = train_custom_binary_comparison(
        norm_retain_tensor, torch.cat([norm_forget_tensor, norm_holdout_tensor]), features_labels)
        binary_results['forget_vs_all'] = train_custom_binary_comparison(
            norm_forget_tensor, torch.cat([norm_retain_tensor, norm_holdout_tensor]), features_labels)
        binary_results['holdout_vs_all'] = train_custom_binary_comparison(
            norm_holdout_tensor, torch.cat([norm_retain_tensor, norm_forget_tensor]), features_labels)
        
        # Extract method name from model name
        method_name = "Unknown"
        for method_key, method_full in UNLEARNING_METHODS.items():
            if method_key in model_name.lower():
                method_name = method_full
                break
        if "simnpo" in model_name.lower():
            method_name = "SimNPO"
            
        max_new_tokens=300
        prompt_type = identify_prompt_column_type(forget_data[0])
        forget_data = forget_data.map(lambda ex: filter_examples_by_num_tokens(ex, prompt_type, tokenizer, max_new_tokens=max_new_tokens))
        forget_data = forget_data.filter(lambda ex: ex['valid_example'])
        
        prompt_type = identify_prompt_column_type(retain_data[0])
        retain_data = retain_data.map(lambda ex: filter_examples_by_num_tokens(ex, prompt_type, tokenizer, max_new_tokens=max_new_tokens))
        retain_data = retain_data.filter(lambda ex: ex['valid_example'])

        prompt_type = identify_prompt_column_type(holdout_data[0])
        holdout_data = holdout_data.map(lambda ex: filter_examples_by_num_tokens(ex, prompt_type, tokenizer, max_new_tokens=max_new_tokens))
        holdout_data = holdout_data.filter(lambda ex: ex['valid_example'])

        # Compute baseline metrics
        print("Computing baseline metrics...")
        zlib_results = compute_zlib_compression_auc(forget_data, retain_data, holdout_data, model, tokenizer)
        
        # Clean up memory after each iteration
        cleanup_gpu_memory()
            
        min_k_results = compute_min_k_percent_auc(forget_data, retain_data, holdout_data, model, tokenizer)
        
        # Clean up memory after each iteration
        cleanup_gpu_memory()
            
        rouge_results = compute_rouge_l_f1_auc(forget_data, retain_data, holdout_data, model, tokenizer)
        
        # Clean up memory after each iteration
        cleanup_gpu_memory()

        # Compile results
        evaluation_results = {
            'model_name': model_name,
            'benchmark': benchmark_name,
            'method': method_name,
            'multi_class': {
                'logistic': {
                    'accuracy': results['multi_class']['logistic']['accuracy'],
                    'f1': results['multi_class']['logistic']['f1'],
                    'roc_auc': results['multi_class']['logistic']['roc_auc']
                },
                'random_forest': {
                    'accuracy': results['multi_class']['random_forest']['accuracy'],
                    'f1': results['multi_class']['random_forest']['f1'], 
                    'roc_auc': results['multi_class']['random_forest']['roc_auc']
                }
            },
            'binary_comparisons': binary_results,
            'feature_importance': feature_importance_results,
            'baselines': {
                            'zlib_compression': zlib_results,
                            'min_k_percent': min_k_results,
                            'rouge_l_f1': rouge_results
                        },
        }
        
        print(f"✅ Evaluation completed for {model_name} on {benchmark_name}")
        print(f"   Logistic Accuracy: {results['multi_class']['logistic']['accuracy']:.3f}")
        print(f"   Random Forest Accuracy: {results['multi_class']['random_forest']['accuracy']:.3f}")
        
        # Clean up GPU memory
        del model, tokenizer
        # Clean up memory after each iteration
        cleanup_gpu_memory()
            
        return evaluation_results
        
    except Exception as e:
        print(f"❌ Error evaluating {model_name} on {benchmark_name}: {e}")
        # Clean up GPU memory on error
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return None

print("Model evaluation function defined")

Model evaluation function defined


In [29]:
def run_comprehensive_experiments(max_models_per_family=3, subset_size=25):
    """
    Run comprehensive experiments across all model families and benchmarks
    
    Args:
        max_models_per_family: Maximum number of models to test per family (for time constraints)
        subset_size: Number of examples per dataset split
        
    Returns:
        list: All evaluation results
    """
    print(f"\n🚀 Starting comprehensive experiments...")
    print(f"   Max models per family: {max_models_per_family}")
    print(f"   Subset size: {subset_size}")
    
    all_results = []
    
    # Define model-benchmark mappings
    model_benchmark_mapping = {
        # WMDP models -> WMDP benchmark
        **{model: "WMDP" for model in MODEL_CONFIGS["wmdp_models"][:max_models_per_family]},
        # **{model: "WMDP" for model in MODEL_CONFIGS["wmdp_zephyr"][:max_models_per_family]},
        
        # TOFU models -> TOFU benchmark  
        **{model: "TOFU" for model in MODEL_CONFIGS["tofu_models"][:max_models_per_family]},
        
        # MUSE models -> MUSE benchmark
        **{model: "MUSE" for model in MODEL_CONFIGS["muse_models"][:max_models_per_family]},
    }
    
    print(f"Total model-benchmark combinations: {len(model_benchmark_mapping)}")
    
    # Run evaluations
    for i, (model_name, benchmark_name) in enumerate(model_benchmark_mapping.items(), 1):
        
        if DEBUG and i >= 3:
            print("Debug mode: stopping after 2 evaluations")
            break
        print(f"\n📊 Progress: {i}/{len(model_benchmark_mapping)}")
        
        # Evaluate model on benchmark
        try:
            result = evaluate_model_on_benchmark(model_name, benchmark_name, subset_size)
        except Exception as e:
            print(f"❌ Exception during evaluation: {e}")
            result = None
            raise e
        if result is not None:
            all_results.append(result)
        
        # Clean up memory after each iteration
        cleanup_gpu_memory()
        
        # Optional: Save intermediate results
        if i % 3 == 0:
            print(f"💾 Saving intermediate results after {i} evaluations...")
            pd.DataFrame(all_results).to_csv('intermediate_results.csv', index=False)
    
    print(f"\n🎉 Comprehensive experiments completed!")
    print(f"   Successful evaluations: {len(all_results)}/{len(model_benchmark_mapping)}")
    
    return all_results

# Create results storage directory
results_dir = "comprehensive_results"
exp_time = datetime.now().strftime("%Y-%m-%d_%H-%M")
SUBSET_SIZE = 10
results_dir = os.path.join(Unlearn_Simple_DIR, results_dir, f"subset_size-{SUBSET_SIZE}", exp_time)
os.makedirs(results_dir, exist_ok=True)

print(f"Results will be saved to: {results_dir}")

Results will be saved to: /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/Unlearn-Simple/comprehensive_results/subset_size-10/2025-09-14_20-36


In [30]:
def analyze_results_and_generate_tables(all_results):
    """
    Analyze experiment results and generate tables matching the paper format
    
    Args:
        all_results: List of evaluation results from experiments
        
    Returns:
        dict: Formatted tables and analysis
    """
    print(f"\n📈 Analyzing results from {len(all_results)} evaluations...")
    
    # Convert results to DataFrame for easier analysis
    results_df = pd.DataFrame(all_results)
    
    # Extract metrics for each classifier type
    detailed_results = []
    for _, row in results_df.iterrows():
        # Logistic Regression results
        detailed_results.append({
            'Model': row['model_name'].split('/')[-1],  # Get model name
            'Benchmark': row['benchmark'],
            'Method': row['method'],
            'Classifier': 'LogReg',
            'AUC': row['multi_class']['logistic']['roc_auc'],
            'Accuracy': row['multi_class']['logistic']['accuracy'], 
            'F1': row['multi_class']['logistic']['f1'],
            'Retain_vs_All': row['binary_comparisons'].get('retain_vs_forget', {}).get('logistic', {}).get('roc_auc', 0.5),
            'Forget_vs_All': row['binary_comparisons'].get('forget_vs_holdout', {}).get('logistic', {}).get('roc_auc', 0.5),
        })
        
        # Random Forest results
        detailed_results.append({
            'Model': row['model_name'].split('/')[-1],
            'Benchmark': row['benchmark'], 
            'Method': row['method'],
            'Classifier': 'Tree',
            'AUC': row['multi_class']['random_forest']['roc_auc'],
            'Accuracy': row['multi_class']['random_forest']['accuracy'],
            'F1': row['multi_class']['random_forest']['f1'],
            'Retain_vs_All': row['binary_comparisons'].get('retain_vs_forget', {}).get('random_forest', {}).get('roc_auc', 0.5),
            'Forget_vs_All': row['binary_comparisons'].get('forget_vs_holdout', {}).get('random_forest', {}).get('roc_auc', 0.5),
        })
    
    detailed_df = pd.DataFrame(detailed_results)
    
    # Generate Table 2: Aggregate Comparison (EPP-UE vs baselines)
    print("\n📊 Generating Table 2: Aggregate Comparison...")
    
    # For this, we'll use our EPP-UE (ILL-based) results vs dummy baseline values
    epp_ue_results = detailed_df.groupby('Classifier').agg({
        'AUC': 'mean',
        'Retain_vs_All': 'mean', 
        'Forget_vs_All': 'mean'
    }).round(3)
    
    # Extract baseline results from all_results
    baseline_agg = {'zlib_compression': [], 'min_k_percent': [], 'rouge_l_f1': []}
    for result in all_results:
        if 'baselines' in result:
            for method, scores in result['baselines'].items():
                baseline_agg[method].append(scores)

    # Compute averages with updated keys
    zlib_avg = {k: np.mean([r[k] for r in baseline_agg['zlib_compression']]) for k in ['retain_vs_all_auc', 'forget_vs_all_auc', 'multi_class_auc']}
    min_k_avg = {k: np.mean([r[k] for r in baseline_agg['min_k_percent']]) for k in ['retain_vs_all_auc', 'forget_vs_all_auc', 'multi_class_auc']}
    rouge_avg = {k: np.mean([r[k] for r in baseline_agg['rouge_l_f1']]) for k in ['retain_vs_all_auc', 'forget_vs_all_auc', 'multi_class_auc']}

    # Update Table 2 data with real values
    table2_data = {
        'Method': ['Zlib Compression', 'MIN-K%++', 'ROUGE-L F1', 'EPP-UE - LogReg (ours)', 'EPP-UE - Tree (ours)'],
        'Retained_vs_Forgotten_AUC': [zlib_avg['retain_vs_all_auc'], min_k_avg['retain_vs_all_auc'], rouge_avg['retain_vs_all_auc'], epp_ue_results.loc['LogReg', 'Retain_vs_All'], epp_ue_results.loc['Tree', 'Retain_vs_All']],
        'Forgotten_vs_Holdout_AUC': [zlib_avg['forget_vs_all_auc'], min_k_avg['forget_vs_all_auc'], rouge_avg['forget_vs_all_auc'], epp_ue_results.loc['LogReg', 'Forget_vs_All'], epp_ue_results.loc['Tree', 'Forget_vs_All']],
        'Overall': [zlib_avg['multi_class_auc'], min_k_avg['multi_class_auc'], rouge_avg['multi_class_auc'], epp_ue_results.loc['LogReg', 'AUC'], epp_ue_results.loc['Tree', 'AUC']]
    }
    table2_df = pd.DataFrame(table2_data)
    
    # Generate Table 3: Model Family Comparison  
    print("📊 Generating Table 3: Model Family Comparison...")
    
    # Group by model family (inferred from model names)
    def get_model_family(model_name):
        if 'llama-3' in model_name.lower():
            return 'LLaMA-3-8B'
        elif 'llama-2' in model_name.lower():
            return 'LLaMA-2-7B'
        elif 'zephyr' in model_name.lower():
            return 'Zephyr-7B'
        else:
            return 'Other'
    
    detailed_df['Model_Family'] = detailed_df['Model'].apply(get_model_family)
    
    # Aggregate by model family and method
    family_results = detailed_df.groupby(['Model_Family', 'Method', 'Classifier']).agg({
        'AUC': 'mean',
        'Accuracy': 'mean',
        'F1': 'mean', 
        'Retain_vs_All': 'mean',
        'Forget_vs_All': 'mean'
    }).round(3)
    
    # Generate Table 4: Detailed Individual Results
    print("📊 Generating Table 4: Detailed Individual Results...")
    
    # Use the detailed_df as the base for Table 4
    table4_df = detailed_df.copy()
    # For AUC at 1% FP, use the computed value from the 'retain_vs_forget' binary comparison
    table4_df['AUC_at_0.1_FP'] = table4_df.apply(
        lambda row: row['binary_comparisons'].get('retain_vs_forget').get(
            row['Classifier'].lower()).get(
                'auc_at_0_1_fp'),  # Fallback to approximate if not available
        axis=1
    )

    # For Multi_class, use the multi-class AUC directly
    table4_df['Multi_class'] = table4_df['AUC']
    
    return {
        'table2': table2_df,
        'table3': family_results,
        'table4': table4_df,
        'detailed_results': detailed_df,
        'summary_stats': {
            'total_evaluations': len(all_results),
            'unique_models': detailed_df['Model'].nunique(),
            'unique_benchmarks': detailed_df['Benchmark'].nunique(),
            'avg_auc_logreg': detailed_df[detailed_df['Classifier']=='LogReg']['AUC'].mean(),
            'avg_auc_tree': detailed_df[detailed_df['Classifier']=='Tree']['AUC'].mean()
        }
    }
    
def display_tables(analysis_results):
    """Display the generated tables in a formatted way"""
    
    print("\n" + "="*80)
    print("TABLE 2: AGGREGATE COMPARISON OF UNLEARNING EVALUATION METRICS")
    print("="*80)
    print(analysis_results['table2'].to_string(index=False))
    
    print("\n" + "="*80)
    print("TABLE 3: AGGREGATE COMPARISON ACROSS MODEL FAMILIES") 
    print("="*80)
    print(analysis_results['table3'].head(20).to_string())
    
    print("\n" + "="*80)
    print("TABLE 4: DETAILED SCORES FOR INDIVIDUAL MODELS (First 10)")
    print("="*80)
    table4_display = analysis_results['table4'][['Model', 'Benchmark', 'Method', 'Classifier', 'AUC', 'Accuracy', 'F1']].head(10)
    print(table4_display.to_string(index=False))
    
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)
    for key, value in analysis_results['summary_stats'].items():
        print(f"{key}: {value}")
    
    return analysis_results

print("Analysis and table generation functions defined")

Analysis and table generation functions defined


In [31]:
assert False

AssertionError: 

In [ ]:
# Run the comprehensive experiments
print("🚀 Starting comprehensive evaluation across all models and benchmarks...")
print("This may take some time depending on available computational resources.")

# Configure experiment parameters
MAX_MODELS_PER_FAMILY = 5  # Limit for computational efficiency

print(f"Configuration:")
print(f"  Max models per family: {MAX_MODELS_PER_FAMILY}")
print(f"  Dataset subset size: {SUBSET_SIZE}")
print(f"  Expected total evaluations: ~{MAX_MODELS_PER_FAMILY * 3 * 2}")  # families * benchmarks * classifiers

# Run experiments
all_results = run_comprehensive_experiments(
    max_models_per_family=MAX_MODELS_PER_FAMILY,
    subset_size=SUBSET_SIZE
)

🚀 Starting comprehensive evaluation across all models and benchmarks...
This may take some time depending on available computational resources.
Configuration:
  Max models per family: 5
  Dataset subset size: 10
  Expected total evaluations: ~30

🚀 Starting comprehensive experiments...
   Max models per family: 5
   Subset size: 10
Total model-benchmark combinations: 5

📊 Progress: 1/5

Evaluating LLM-GAT/llama-3-8b-instruct-elm-checkpoint-8 on WMDP
Loading model: LLM-GAT/llama-3-8b-instruct-elm-checkpoint-8


Loading checkpoint shards: 100%|██████████| 7/7 [00:09<00:00,  1.38s/it]


Model loaded successfully on device: cuda:0
Loading WMDP data...
WMDP data loaded: 10 forget, 10 retain, 10 holdout
Running ILL evaluation...
working on forget subset
Output file /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/data/LLM-GAT-llama-3-8b-instruct-elm-checkpoint-8/top_k_20_n_tokens_0.1_k_neighbors_15/exp_wmdp_neighbors/forget.json already exists. Skipping generation.
Loaded 10 examples from /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/data/LLM-GAT-llama-3-8b-instruct-elm-checkpoint-8/top_k_20_n_tokens_0.1_k_neighbors_15/exp_wmdp_neighbors/forget.json
Extracting features


Batches: 100%|██████████| 3/3 [00:00<00:00, 24.88it/s]


working on retain subset
Output file /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/data/LLM-GAT-llama-3-8b-instruct-elm-checkpoint-8/top_k_20_n_tokens_0.1_k_neighbors_15/exp_wmdp_neighbors/retain.json already exists. Skipping generation.
Loaded 10 examples from /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/data/LLM-GAT-llama-3-8b-instruct-elm-checkpoint-8/top_k_20_n_tokens_0.1_k_neighbors_15/exp_wmdp_neighbors/retain.json
Extracting features


Batches: 100%|██████████| 3/3 [00:00<00:00, 23.17it/s]


working on holdout subset


In [ ]:
# Analyze results and generate tables
if len(all_results) > 0:
    print(f"\n📊 Analyzing {len(all_results)} experimental results...\n")

    # Generate analysis and tables
    analysis_results = analyze_results_and_generate_tables(all_results)

    # Display formatted tables
    display_tables(analysis_results)

    # Save results to files
    print(f"\n💾 Saving results to files...\n")

    # Save each table as a separate CSV and print absolute path
    table2_path = os.path.abspath(f"{results_dir}/table2_aggregate_comparison.csv")
    table3_path = os.path.abspath(f"{results_dir}/table3_model_family_comparison.csv")
    table4_path = os.path.abspath(f"{results_dir}/table4_detailed_results.csv")
    detailed_path = os.path.abspath(f"{results_dir}/all_detailed_results.csv")
    json_path = os.path.abspath(f"{results_dir}/complete_results.json")

    analysis_results['table2'].to_csv(table2_path, index=False)
    print(f"Table 2 saved to: {table2_path}")

    analysis_results['table3'].to_csv(table3_path)
    print(f"Table 3 saved to: {table3_path}")

    analysis_results['table4'].to_csv(table4_path, index=False)
    print(f"Table 4 saved to: {table4_path}")

    analysis_results['detailed_results'].to_csv(detailed_path, index=False)
    print(f"All detailed results saved to: {detailed_path}")

    # Save complete results as JSON for further analysis
    import json

    def make_json_serializable(obj):
        if isinstance(obj, (dict)):
            return {k: make_json_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, (list, tuple)):
            return [make_json_serializable(v) for v in obj]
        elif isinstance(obj, (str, int, float, bool, type(None))):
            return obj
        else:
            return str(obj)

    with open(json_path, 'w') as f:
        serializable_results = [make_json_serializable(result) for result in all_results]
        json.dump({
            'experiment_results': serializable_results,
            'summary_statistics': make_json_serializable(analysis_results['summary_stats']),
            'configuration': {
                'max_models_per_family': MAX_MODELS_PER_FAMILY,
                'subset_size': SUBSET_SIZE
            }
        }, f, indent=2)
    print(f"Complete results JSON saved to: {json_path}")

else:
    print("❌ No results to analyze. Check if experiments ran successfully.")

❌ No results to analyze. Check if experiments ran successfully.


: 

: 

In [ ]:
# Create visualizations and final summary
if len(all_results) > 0:
    print(f"\n📈 Creating visualizations...")
    
    # Create performance comparison plots
    plt.figure(figsize=(15, 10))
    
    # Plot 1: AUC comparison by model family
    plt.subplot(2, 2, 1)
    detailed_df = analysis_results['detailed_results']
    sns.boxplot(data=detailed_df, x='Model_Family', y='AUC', hue='Classifier')
    plt.title('AUC Performance by Model Family')
    plt.xticks(rotation=45)
    plt.legend(title='Classifier')
    
    # Plot 2: Benchmark comparison
    plt.subplot(2, 2, 2)
    sns.boxplot(data=detailed_df, x='Benchmark', y='AUC', hue='Classifier')
    plt.title('AUC Performance by Benchmark')
    plt.legend(title='Classifier')
    
    # Plot 3: Method comparison
    plt.subplot(2, 2, 3)
    method_means = detailed_df.groupby(['Method', 'Classifier'])['AUC'].mean().reset_index()
    sns.barplot(data=method_means, x='Method', y='AUC', hue='Classifier')
    plt.title('Average AUC by Unlearning Method')
    plt.xticks(rotation=45)
    plt.legend(title='Classifier')
    
    # Plot 4: Overall performance distribution
    plt.subplot(2, 2, 4)
    plt.hist(detailed_df[detailed_df['Classifier']=='LogReg']['AUC'], alpha=0.7, label='LogReg', bins=10)
    plt.hist(detailed_df[detailed_df['Classifier']=='Tree']['AUC'], alpha=0.7, label='Tree', bins=10)
    plt.xlabel('AUC Score')
    plt.ylabel('Frequency')
    plt.title('Distribution of AUC Scores')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(f'{results_dir}/performance_overview.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n🎯 FINAL SUMMARY")
    print(f"="*60)
    print(f"Comprehensive Unlearning Evaluation Results:")
    print(f"  • Total models evaluated: {analysis_results['summary_stats']['unique_models']}")
    print(f"  • Benchmarks tested: {analysis_results['summary_stats']['unique_benchmarks']}")
    print(f"  • Total evaluations: {analysis_results['summary_stats']['total_evaluations']}")
    print(f"  • Average AUC (LogReg): {analysis_results['summary_stats']['avg_auc_logreg']:.3f}")
    print(f"  • Average AUC (Tree): {analysis_results['summary_stats']['avg_auc_tree']:.3f}")
    
    # Best performing combinations
    best_logreg = detailed_df[detailed_df['Classifier']=='LogReg'].nlargest(1, 'AUC')
    best_tree = detailed_df[detailed_df['Classifier']=='Tree'].nlargest(1, 'AUC')
    
    print(f"\n🏆 BEST PERFORMING COMBINATIONS:")
    print(f"  Logistic Regression:")
    if not best_logreg.empty:
        row = best_logreg.iloc[0]
        print(f"    {row['Model']} on {row['Benchmark']} ({row['Method']}) - AUC: {row['AUC']:.3f}")
    
    print(f"  Random Forest:")
    if not best_tree.empty:
        row = best_tree.iloc[0]
        print(f"    {row['Model']} on {row['Benchmark']} ({row['Method']}) - AUC: {row['AUC']:.3f}")
    
    print(f"\n📊 The tables above replace the dummy values in the original LaTeX tables.")
    print(f"💾 All results and visualizations saved to: {results_dir}/")
    print(f"🎉 Comprehensive evaluation completed successfully!")
    
else:
    print("❌ No results available for visualization and summary.")

❌ No results available for visualization and summary.


: 

: 

In [ ]:
# Optional: Quick test with a single model (for debugging/testing)
# Uncomment and run this cell to test the pipeline with just one model

print("🧪 Running quick test with a single model...")
test_model = "LLM-GAT/llama-3-8b-instruct-elm-checkpoint-8"
test_benchmark = "WMDP"

# test_result = evaluate_model_on_benchmark(test_model, test_benchmark, subset_size=10)
# if test_result:
#     print("✅ Quick test successful!")
#     print(f"   Model: {test_result['model_name']}")
#     print(f"   Benchmark: {test_result['benchmark']}")  
#     print(f"   Method: {test_result['method']}")
#     print(f"   LogReg AUC: {test_result['multi_class']['logistic']['roc_auc']:.3f}")
#     print(f"   Tree AUC: {test_result['multi_class']['random_forest']['roc_auc']:.3f}")
# else:
#     print("❌ Quick test failed")

print("💡 Notebook setup complete!")
print("   Run the cells above to execute the comprehensive evaluation.")
print("   Modify MAX_MODELS_PER_FAMILY and SUBSET_SIZE for different experiment scales.")

🧪 Running quick test with a single model...

Evaluating LLM-GAT/llama-3-8b-instruct-elm-checkpoint-8 on WMDP
Loading model: LLM-GAT/llama-3-8b-instruct-elm-checkpoint-8


Loading checkpoint shards: 100%|██████████| 7/7 [00:10<00:00,  1.52s/it]


Model loaded successfully on device: cuda:0
Loading WMDP data...
WMDP data loaded: 10 forget, 10 retain, 10 holdout
Running ILL evaluation...
working on forget subset
Output file /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/data/LLM-GAT-llama-3-8b-instruct-elm-checkpoint-8/top_k_20_n_tokens_0.1_k_neighbors_15/exp_wmdp_neighbors/forget.json already exists. Skipping generation.
Loaded 10 examples from /home/liranc6/W25/adversarial-attacks-on-deep-learning/project/data/LLM-GAT-llama-3-8b-instruct-elm-checkpoint-8/top_k_20_n_tokens_0.1_k_neighbors_15/exp_wmdp_neighbors/forget.json
Extracting features


Ignored error while writing commit hash to /home/liranc6/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/refs/main: [Errno 122] Disk quota exceeded: '/home/liranc6/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/refs/main'.
Batches: 100%|██████████| 3/3 [00:00<00:00, 23.77it/s]


working on retain subset


KeyboardInterrupt: 

: 

: 

A code to run experiments and outputting results for those tables (replacing dummy values):
```latex
\begin{table}[H]
\centering
\renewcommand{\arraystretch}{1.15}
\makecell{\textbf{Table 2: Aggregate Comparison of Unlearning Evaluation Metrics Across Methods}}\\[1ex]
\begin{tabular}{lccc}
\hline
\textbf{Method} & \textbf{Retained vs. Forgotten (AUC)} & \textbf{Forgotten vs. Holdout (AUC)} & \textbf{Overall} \\
\hline
QA-based score & 0.62 & 0.58 & 0.60 \\
Zlib Compression & 0.65 & 0.61 & 0.63 \\
MIN-K\%++ & 0.68 & 0.64 & 0.66 \\
ROUGE-L F1 & 0.60 & 0.55 & 0.57 \\
\textbf{EPP-UE (ours)} & \textbf{0.81} & \textbf{0.77} & \textbf{0.79} \\
\hline
\end{tabular}
\caption{Comparison of unlearning evaluation metrics across TOFU benchmark (dummy values).}
\end{table}

\begin{table}[H]
\centering
\renewcommand{\arraystretch}{1.15}
\makecell{\textbf{Table 3: Aggregate Comparison Across Model Families, Methods, and Classifiers}}\\[1ex]
\begin{tabular}{lllccllll}
\hline
\textbf{Model Family} & \textbf{Method} & \makecell{AUC /\\AUC@0.1\%FP} & \textbf{ACC} & \makecell{Retain\\vs All} & \makecell{Forget\\vs All} & \makecell{Multi-\\class} \\
\hline
LLaMA-3-8B   & QA-based      & 0.63 / 0.46 & 0.61 & 0.59 & 0.60 & 0.42 \\
LLaMA-3-8B   & ROUGE-L F1    & 0.63 / 0.46 & 0.61 & 0.59 & 0.60 & 0.42 \\
LLaMA-3-8B   & MIN-K\%++     & 0.63 / 0.46 & 0.61 & 0.59 & 0.60 & 0.42 \\
LLaMA-3-8B   & Zlib          & 0.63 / 0.46 & 0.61 & 0.59 & 0.60 & 0.42 \\
LLaMA-3-8B   & \textbf{EPP-UE - LogReg (ours)} & \textbf{0.82 / 0.71} & \textbf{0.80} & \textbf{0.78} & \textbf{0.79} & \textbf{0.66} \\
LLaMA-3-8B   & \textbf{EPP-UE - Tree (ours)}   & \textbf{0.84 / 0.73} & \textbf{0.81} & \textbf{0.79} & \textbf{0.80} & \textbf{0.68} \\
LLaMA-2-7B   & QA-based      & 0.63 / 0.46 & 0.61 & 0.59 & 0.60 & 0.42 \\
LLaMA-2-7B   & ROUGE-L F1    & 0.63 / 0.46 & 0.61 & 0.59 & 0.60 & 0.42 \\
LLaMA-2-7B   & MIN-K\%++     & 0.63 / 0.46 & 0.61 & 0.59 & 0.60 & 0.42 \\
LLaMA-2-7B   & Zlib          & 0.63 / 0.46 & 0.61 & 0.59 & 0.60 & 0.42 \\
LLaMA-2-7B   & \textbf{EPP-UE - LogReg (ours)} & 0.81 / 0.70 & 0.79 & 0.77 & 0.78 & 0.65 \\
LLaMA-2-7B   & \textbf{EPP-UE - Tree (ours)}   & 0.81 / 0.70 & 0.79 & 0.77 & 0.78 & 0.65 \\
Zephyr-7B    & QA-based      & 0.63 / 0.46 & 0.61 & 0.59 & 0.60 & 0.42 \\
Zephyr-7B    & ROUGE-L F1    & 0.63 / 0.46 & 0.61 & 0.59 & 0.60 & 0.42 \\
Zephyr-7B    & MIN-K\%++     & 0.63 / 0.46 & 0.61 & 0.59 & 0.60 & 0.42 \\
Zephyr-7B    & Zlib          & 0.63 / 0.46 & 0.61 & 0.59 & 0.60 & 0.42 \\
Zephyr-7B    & \textbf{EPP-UE - LogReg (ours)} & \textbf{0.80} / 0.69 & 0.78 & 0.76 & 0.77 & 0.64 \\
Zephyr-7B    & \textbf{EPP-UE - Tree (ours)}   & 0.80 / 0.69 & 0.78 & 0.76 & 0.77 & 0.64 \\
\hline
\end{tabular}
\caption{Aggregate comparison of unlearning evaluation metrics across model families, methods, and classifiers. Scores are averaged over all checkpoints in each family (dummy values).}
\end{table}

\begin{table}[H]
\centering
\renewcommand{\arraystretch}{1.1}
\makecell{\textbf{Table 4: Detailed Scores for Individual Checkpoints and Models}}\\[1ex]
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{l l l l c c c c c c}
\hline
\makecell{\textbf{Classifier}} & \makecell{\textbf{Benchmark}} & \makecell{\textbf{Model}\\\textbf{Index}} & \makecell{\textbf{Unlearning}\\\textbf{Method}} & \makecell{\textbf{AUC}\\} & \makecell{\textbf{AUC@}\\\textbf{0.1\%FP}} & \textbf{F1} & \makecell{\textbf{Retain}\\\textbf{vs All}} & \makecell{\textbf{Forget}\\\textbf{vs All}} & \makecell{\textbf{Multi-}\\\textbf{class}} \\
\hline
LogReg & TOFU & M1 & GradEdit & 0.81 & 0.71 & 0.78 & 0.77 & 0.78 & 0.66 \\
Tree   & TOFU & M1 & GradEdit & 0.84 & 0.74 & 0.81 & 0.80 & 0.81 & 0.69 \\
LogReg & TOFU & M2 & FineTune & 0.82 & 0.72 & 0.79 & 0.78 & 0.79 & 0.67 \\
Tree   & TOFU & M2 & FineTune & 0.85 & 0.75 & 0.82 & 0.81 & 0.82 & 0.70 \\
LogReg & MUSE & M3 & Scrubbing & 0.78 & 0.68 & 0.76 & 0.74 & 0.75 & 0.62 \\
Tree   & MUSE & M3 & Scrubbing & 0.80 & 0.70 & 0.78 & 0.76 & 0.77 & 0.64 \\
LogReg & WMDP & M4 & GradEdit & 0.76 & 0.66 & 0.74 & 0.72 & 0.73 & 0.60 \\
Tree   & WMDP & M4 & GradEdit & 0.78 & 0.68 & 0.76 & 0.74 & 0.75 & 0.62 \\
LogReg & TOFU & M5 & FineTune & 0.79 & 0.69 & 0.77 & 0.75 & 0.76 & 0.63 \\
Tree   & TOFU & M5 & FineTune & 0.81 & 0.71 & 0.79 & 0.77 & 0.78 & 0.65 \\
....
\hline
\end{tabular}
\end{adjustbox}
\caption{
All experiments and evaluations presented in the tables were conducted using our proposed EPP-UE method. Detailed scores for individual checkpoints and models (dummy values).\\
Model indices: 
M1 = llama-2-7b-chat, 
M2 = llama-3-8b, 
M3 = zephyr-7b-beta, 
M4 = phi-3.5, 
M5 = gemma-7b.\\
Benchmarks: TOFU, MUSE, WMDP.\\
Unlearning methods: GradEdit = gradient editing, FineTune = fine-tuning removal, Scrubbing = approximate scrubbing.
}
\end{table}
```


I have those models and unlearning methods: 
"LLM-GAT/llama-3-8b-instruct-{elm, graddiff, tar, pbj, rmu-lat, rmu, rr, repnoise}-checkpoint-8" (for WMDP), "OPTML-Group/SimNPO-TOFU-forget{05, 10}-Llama-2-7b-chat", "OPTML-Group/SimNPO-MUSE-{News, Books}-iclm-7b" which are llama-2-7B-chat. and "OPTML-Group/SimNPO-WMDP-zephyr-7b-beta"